# @examples/windpowerlib

The windpowerlib is a library to model the output of wind turbines and farms.

Run all cells: **Runtime → Run all** (or ⌘/Ctrl + F9). The last code cell prints the endpoint address.


In [ ]:
PRIMITIVE = "@examples/windpowerlib"
ARCHIVE = "https://hub-api.powerai.ai/api/artifacts/examples/windpowerlib/download-archive/"
WORKDIR = "/content/primitive"

import shutil, urllib.request, zipfile
from pathlib import Path

shutil.rmtree(WORKDIR, ignore_errors=True)          # re-runs start clean
Path(WORKDIR).mkdir(parents=True, exist_ok=True)

archive = Path("/content/primitive.zip")
urllib.request.urlretrieve(ARCHIVE, archive)
with zipfile.ZipFile(archive) as zf:
    zf.extractall(WORKDIR)

files = sorted(p for p in Path(WORKDIR).rglob("*") if p.is_file())
print(f"{PRIMITIVE}: {len(files)} files in {WORKDIR}")
for p in files[:20]:
    print(" ", p.relative_to(WORKDIR))
if len(files) > 20:
    print(f"  … and {len(files) - 20} more")


In [ ]:
import importlib.metadata as md, os, subprocess, sys
from pathlib import Path

print(f"Python {sys.version.split()[0]}")

# PY is the interpreter the later cells install into and serve from. It is Colab's own unless
# this primitive asks for another Python (WANT, empty when it does not say): then a virtual
# environment on that Python is made with uv, which fetches the interpreter itself.
PY = sys.executable
WANT = ""
if WANT and not sys.version.startswith(WANT + "."):
    VENV = "/content/venv"
    try:
        # --clear: running the notebook again in the same session finds the environment of the
        # first run, and uv refuses to make one over it.
        for step in (
            [sys.executable, "-m", "pip", "install", "-q", "uv"],
            [sys.executable, "-m", "uv", "venv", "--clear", "--seed", "--python", WANT, VENV],
        ):
            subprocess.run(step, check=True, capture_output=True, text=True)
        PY = VENV + "/bin/python"
        # Colab sets PYTHONPATH to its own packages, built for its own Python.
        os.environ.pop("PYTHONPATH", None)
        print(f"This primitive asks for Python {WANT}: using it in {VENV}")
    except subprocess.CalledProcessError as exc:
        print(f"Could not set up Python {WANT}; staying on {sys.version.split()[0]}:")
        print((exc.stderr or "")[-600:])

def declared():
    # A fresh environment has none of what Colab ships (numpy, pandas ...). A requirements.txt
    # that leans on Colab for them (ANDES) or is absent (pvlib) leaves the endpoint unable to
    # import, so pyproject's dependencies go in too. Colab's own Python needs nothing added.
    if PY == sys.executable:
        return []
    try:
        import tomllib
        table = tomllib.loads(Path(WORKDIR, "pyproject.toml").read_text())
        return list(table.get("project", {}).get("dependencies", []))
    except Exception:
        return []


# The file nearest the root first: a requirements.txt under docs/ or tests/ is not this
# primitive's, and rglob alone would pick whichever it met first.
found = sorted(
    Path(WORKDIR).rglob("requirements.txt"),
    key=lambda p: (len(p.relative_to(WORKDIR).parts), str(p)),
)
if not found:
    print("No requirements.txt. Install what you need with pip, e.g. !pip install pandas")
    if PY != sys.executable:
        done = subprocess.run(
            [PY, "-m", "pip", "install", "-q", "fastapi", "uvicorn", *declared()],
            capture_output=True, text=True,
        )
        if done.returncode != 0:
            print("pip failed:")
            print(done.stderr[-2000:])
else:
    req = found[0]
    print(f"Installing {req.relative_to(WORKDIR)} …")
    if len(found) > 1:
        others = ", ".join(str(p.relative_to(WORKDIR)) for p in found[1:4])
        print(f"Not installed (only one file is): {others}")

    def versions():
        return {d.metadata["Name"].lower(): d.version for d in md.distributions() if d.metadata["Name"]}

    def install():
        # The endpoint needs FastAPI and uvicorn, which Colab ships and a fresh environment does
        # not. Asked for in the same install, pip picks versions that fit the primitive's pins.
        serve = ["fastapi", "uvicorn"] if PY != sys.executable else []
        return subprocess.run(
            [PY, "-m", "pip", "install", "-q", "-r", str(req), *serve, *declared()],
            capture_output=True, text=True,
        )

    before = versions()
    done = install()
    if done.returncode != 0 and PY != sys.executable:
        # Keep the requested Python only if it works: fall back to Colab's, and say so.
        print(f"pip failed on Python {WANT}; trying Colab's {sys.version.split()[0]} instead.")
        PY = sys.executable
        done = install()
    if done.returncode == 0:
        print("done")
        after = versions()
        # An install can replace what Colab ships; say so, so a later surprise has a cause.
        for name in sorted(n for n in before if n in after and before[n] != after[n])[:12]:
            print(f"  changed: {name} {before[name]} -> {after[name]}")
    else:
        print(
            "pip failed. pip installs the whole file or none of it, so one package that "
            "cannot be built on this Python stops all of it:"
        )
        print(done.stderr[-2000:])


In [ ]:
from pathlib import Path

# PowerAI's server shell: it starts the primitive's endpoint, answers /health with its state
# while it loads, and stops a call that runs past 90 s from taking the whole server with it.
Path(WORKDIR, "powerai_serve.py").write_text('''"""PowerAI serve shell: the server the launcher starts in front of a primitive's endpoint file.

The endpoint file (powerai_endpoint.py, or the Hub's model or dataset runner) is the app that
answers /run. This shell is what uvicorn serves, and it does three things the file cannot:

* Start-up. It imports the file and sends it the file's own SAMPLE_INPUTS, once, in the
  background, so a primitive that downloads or loads weights on its first call does that before
  anyone asks. GET /health says "warming" until then, then "ok" (or "failed", with the reason).
  The launcher waits for "ok" before it prints the address.
* A time limit. A call runs in a worker thread, not on the server's event loop, so a slow call
  cannot stop /health or other requests from answering. After RUN_SECONDS the caller gets a
  504 "timeout" reply instead of a tunnel error. The call itself cannot be stopped and runs on.
* A cap of MAX_RUNNING calls at once, so retries of a slow call cannot pile up in memory.

Written for any Python the launcher may pick, down to 3.8: standard library only.
"""

import asyncio
import importlib
import json
import logging
import os
import threading
import time

MODULE = os.environ.get("POWERAI_ENDPOINT_MODULE", "powerai_endpoint")
# Under the 100 s after which Cloudflare's quick tunnel gives up on a request.
RUN_SECONDS = float(os.environ.get("POWERAI_RUN_SECONDS", "90"))
MAX_RUNNING = 2

log = logging.getLogger("powerai_serve")


class State:
    status = "warming"  # warming | ok | failed
    detail = ""
    problems = []  # start-up calls that did not answer 200: the file is up, but its tool is not working
    app = None
    running = 0
    started = time.time()


state = State()


def _say(text):
    print("powerai_serve: " + text, flush=True)


def _error(code, message):
    return {"powerai": "1", "ok": False, "error": {"code": code, "message": message}}


async def _respond(send, status, data):
    body = data if isinstance(data, bytes) else json.dumps(data).encode()
    await send(
        {
            "type": "http.response.start",
            "status": status,
            "headers": [
                (b"content-type", b"application/json"),
                (b"content-length", str(len(body)).encode()),
            ],
        }
    )
    await send({"type": "http.response.body", "body": body})


async def _read_body(receive):
    chunks = []
    while True:
        message = await receive()
        if message["type"] != "http.request":
            break
        chunks.append(message.get("body", b""))
        if not message.get("more_body"):
            break
    return b"".join(chunks)


async def _post(app, path, body):
    """POST `body` to the endpoint file's own app, in this event loop. Returns (status, bytes)."""
    pending = [{"type": "http.request", "body": body, "more_body": False}]
    done = asyncio.Event()
    out = {"status": 500, "body": b""}

    async def receive():
        if pending:
            return pending.pop()
        await done.wait()
        return {"type": "http.disconnect"}

    async def send(message):
        if message["type"] == "http.response.start":
            out["status"] = message["status"]
        elif message["type"] == "http.response.body":
            out["body"] += message.get("body", b"")
            if not message.get("more_body"):
                done.set()

    scope = {
        "type": "http",
        "asgi": {"version": "3.0"},
        "http_version": "1.1",
        "method": "POST",
        "scheme": "http",
        "path": path,
        "raw_path": path.encode(),
        "query_string": b"",
        "root_path": "",
        "headers": [
            (b"content-type", b"application/json"),
            (b"content-length", str(len(body)).encode()),
        ],
        "client": ("127.0.0.1", 0),
        "server": ("127.0.0.1", 0),
    }
    await app(scope, receive, send)
    return out["status"], out["body"]


def _work(body):
    """Runs in a worker thread, with an event loop of its own: the app may block that loop."""
    return asyncio.run(_post(state.app, "/run", body))


def _warm():
    try:
        module = importlib.import_module(MODULE)
        state.app = module.app
        samples = getattr(module, "SAMPLE_INPUTS", None)
        for tool, arguments in (samples if isinstance(samples, dict) else {}).items():
            body = json.dumps({"powerai": "1", "tool": tool, "input": arguments}).encode()
            began = time.time()
            status, reply = _work(body)
            _say(f"start-up call to {tool} answered {status} in {time.time() - began:.0f} s")
            if status != 200:
                _say("its reply: " + reply[:500].decode(errors="replace"))
                try:
                    why = json.loads(reply)["error"]["message"]
                except Exception:
                    why = reply[:200].decode(errors="replace")
                state.problems.append(f"{tool} answered {status}: {why}")
    except BaseException as exc:  # a file that exits at import must not leave "warming" forever
        log.exception("could not start %s", MODULE)
        state.status, state.detail = "failed", f"{type(exc).__name__}: {exc}"
        return
    state.status = "ok"
    _say(f"ready after {time.time() - state.started:.0f} s")


def _release(future):
    state.running -= 1
    if not future.cancelled():
        future.exception()  # a call that timed out is not awaited any more: mark it seen


async def _run(receive, send):
    if state.status == "warming":
        waited = int(time.time() - state.started)
        message = f"Still loading the primitive ({waited} s so far). Try again in a minute."
        return await _respond(send, 503, _error("unavailable", message))
    if state.status == "failed":
        message = "The primitive did not start: " + state.detail
        return await _respond(send, 503, _error("unavailable", message))
    body = await _read_body(receive)
    if state.running >= MAX_RUNNING:
        message = f"{MAX_RUNNING} calls are already running; try again when one has finished."
        return await _respond(send, 503, _error("unavailable", message))
    state.running += 1
    future = asyncio.get_running_loop().run_in_executor(None, _work, body)
    future.add_done_callback(_release)
    try:
        status, reply = await asyncio.wait_for(asyncio.shield(future), RUN_SECONDS)
    except asyncio.TimeoutError:  # noqa: UP041 (the builtin one only from Python 3.11)
        message = (
            f"The primitive did not answer within {RUN_SECONDS:.0f} s; it may still be running."
        )
        return await _respond(send, 504, _error("timeout", message))
    except Exception:
        log.exception("the call failed")
        return await _respond(send, 500, _error("internal", "The call failed."))
    await _respond(send, status, reply)


async def app(scope, receive, send):
    if scope["type"] == "lifespan":
        while True:
            message = await receive()
            if message["type"] == "lifespan.startup":
                threading.Thread(target=_warm, daemon=True).start()
                await send({"type": "lifespan.startup.complete"})
            elif message["type"] == "lifespan.shutdown":
                await send({"type": "lifespan.shutdown.complete"})
                return
    if scope["type"] != "http":
        return
    route = (scope["method"], scope["path"])
    if route == ("GET", "/health"):
        data = {"status": state.status}
        if state.detail:
            data["error"] = state.detail
        if state.problems:
            data["startup_problems"] = state.problems
        return await _respond(send, 200, data)
    if route == ("POST", "/run"):
        return await _run(receive, send)
    await _respond(send, 404, _error("not_found", "No such route."))
''')


In [ ]:
import json, os, re, subprocess, sys, time, urllib.request

PY = globals().get("PY") or sys.executable  # the install cell's choice, if this session has it
PORT = 8011
MODULE = "powerai_endpoint"
SHELL = "powerai_serve"  # the cell above wrote it: warm-up, /health, time limit
CLOUDFLARED = "/usr/local/bin/cloudflared"

# Re-running this cell has to be safe: a previous run leaves a server and a tunnel
# alive, and the next one would otherwise pass the health probe anyway because the
# *stale* server answers it — a stale process serving stale code, reported as fresh.
for pattern in ("cloudflared tunnel", f"uvicorn {SHELL}:app", f"uvicorn {MODULE}:app"):
    subprocess.run(["pkill", "-f", pattern], capture_output=True)
time.sleep(1)

if not os.path.exists(CLOUDFLARED):
    got = subprocess.run(
        "wget -q https://github.com/cloudflare/cloudflared/releases/latest/download/"
        f"cloudflared-linux-amd64 -O {CLOUDFLARED} && chmod +x {CLOUDFLARED}",
        shell=True, capture_output=True, text=True,
    )
    if got.returncode != 0:
        raise RuntimeError(f"could not install cloudflared:\n{got.stderr[-1000:]}")
    print("cloudflared installed")
else:
    print("cloudflared already present")

# --app-dir puts WORKDIR on the import path for this one process, so the archive
# need not be pip-installed (and may not even be a package) to be served.
# A src/ layout keeps its packages one folder down; both folders go on the path.
# MPLBACKEND: Colab sets it to a notebook backend that only Colab's own Python has, so matplotlib
# will not even import in another environment. A server draws nothing on screen: Agg.
env = dict(os.environ, POWERAI_ENDPOINT_MODULE=MODULE, MPLBACKEND="Agg", PYTHONPATH=os.pathsep.join(
    [WORKDIR, os.path.join(WORKDIR, "src"), os.environ.get("PYTHONPATH", "")]
))
# The server's own output goes to a file, not nowhere: a file that fails on import, or hides an
# error behind a plain "internal" reply, leaves its cause here.
LOG = os.path.join(os.path.dirname(WORKDIR), "endpoint.log")
server = subprocess.Popen(
    [PY, "-m", "uvicorn", f"{SHELL}:app", "--app-dir", WORKDIR,
     "--host", "127.0.0.1", "--port", str(PORT)],
    stdout=open(LOG, "w"), stderr=subprocess.STDOUT, env=env,
)

# The server answers /health at once with "warming" while it loads the primitive (a model's
# weights can take minutes the first time) and says "ok" when a first call has gone through. The
# address is printed only after that, so the first question asked is not the one that waits.
print("Starting the primitive. It loads its code and data first; the first start can take minutes.")
status, problems, began, noted = "", [], time.time(), time.time()
while time.time() - began < 1200:
    if server.poll() is not None:
        break  # it exited: nothing to wait for
    try:
        health = json.load(urllib.request.urlopen(f"http://127.0.0.1:{PORT}/health", timeout=2))
        status, problems = health.get("status", ""), health.get("startup_problems", [])
    except Exception:
        status = ""  # not listening yet
    if status in ("ok", "failed"):
        break
    if time.time() - noted >= 30:
        noted = time.time()
        print(f"  still loading, {int(noted - began)} s")
    time.sleep(2)
if status != "ok":
    try:
        output = open(LOG).read()[-3000:]
    except OSError:
        output = ""
    if status == "warming" and server.poll() is None:
        why = "is still loading after 20 minutes"
    else:
        why = "did not come up as a healthy server"
    raise RuntimeError(
        f"{MODULE}.py {why} on :{PORT}. Its output:\n{output}\n"
        "This is the archive's own code, not the Hub's."
    )
print(f"endpoint up on :{PORT} after {int(time.time() - began)} s")
for problem in problems:  # the server is up, but its own start-up call did not work
    print(f"WARNING: the start-up call failed: {problem}")
print(f"Server log: {LOG}  (after a failed call, run  !tail -n 40 {LOG}  to see why)")

PUBLIC_URL = ""
tunnel = None
tail: list = []
for attempt in range(1, 4):
    if tunnel is not None:
        tunnel.kill()
    tunnel = subprocess.Popen(
        ["cloudflared", "tunnel", "--no-autoupdate", "--url", f"http://127.0.0.1:{PORT}"],
        stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1,
    )
    deadline = time.time() + 60
    while time.time() < deadline:
        line = tunnel.stdout.readline()
        if not line:
            break  # cloudflared exited
        tail = (tail + [line.rstrip()])[-15:]
        found = re.search(r"https://[-a-z0-9]+\.trycloudflare\.com", line)
        if found:
            PUBLIC_URL = found.group(0)
            break
    if PUBLIC_URL:
        break
    print(f"tunnel attempt {attempt} of 3 gave no address")
    time.sleep(3)
if not PUBLIC_URL:
    raise RuntimeError(
        "cloudflared gave no tunnel address after 3 tries. Its last output:\n"
        + "\n".join(tail)
        + "\nThis is Cloudflare's free tunnel service, not the primitive. Re-run this cell in a minute."
    )

ENDPOINT_URL = PUBLIC_URL + "/run"
print()
print("Endpoint address:")
print("   ", ENDPOINT_URL)

PASTE_LINE = f"Run {PRIMITIVE} at {ENDPOINT_URL}"
print()
print("Paste this line into the agent chat:")
print("   ", PASTE_LINE)
try:
    import json
    from IPython.display import HTML, display

    _line = json.dumps(PASTE_LINE).replace("<", chr(92) + "u003c")
    display(HTML(
        '<button id="powerai-copy" style="padding:6px 12px;cursor:pointer">Copy the line</button>'
        ' <span id="powerai-copied"></span>'
        "<script>(function () {"
        " var line = " + _line + ";"
        " var button = document.getElementById('powerai-copy');"
        " var note = document.getElementById('powerai-copied');"
        " function fallback() {"
        "  var box = document.createElement('textarea'); box.value = line;"
        "  document.body.appendChild(box); box.select();"
        "  try { document.execCommand('copy'); note.textContent = ' Copied.'; }"
        "  catch (e) { note.textContent = ' Select the line above and copy it.'; }"
        "  document.body.removeChild(box);"
        " }"
        " button.onclick = function () {"
        "  if (navigator.clipboard && navigator.clipboard.writeText) {"
        "   navigator.clipboard.writeText(line).then(function () { note.textContent = ' Copied.'; }, fallback);"
        "  } else { fallback(); }"
        " };"
        "})();</script>"
    ))
except Exception:
    pass  # the plain line above is enough


Copy the line printed above (use the Copy button, or select it) and paste it into the agent chat on [the Agent tab](https://hub.powerai.ai/examples/windpowerlib/agent).